# 🏗️ Multi-Model Architecture — Drone Detection (Local, Pre-trained Weights)
Loads all 4 trained checkpoints locally and runs evaluation + inference. No training needed.

| # | Model | Weights file | Architecture |
|---|-------|-------------|-------------|
| 1 | **YOLOv8s** | `drone_yolov8s_final.pt` | CSP-Darknet C2f + FPN-PAN |
| 2 | **RT-DETR-L** | `best_detr.pt` | ResNet-50 + Hybrid Transformer |
| 3 | **Faster R-CNN** | `fasterrcnn_drone.pth` | MobileNetV3-Large-320-FPN |
| 4 | **SSD** | `ssd_drone_model_kaggle.pth` | SSDLite MobileNetV3-Large-320 |

> **Classes:** `0: AirPlane` · `1: Drone` · `2: Helicopter`


## 1 · Imports

In [32]:
import os, glob, random, time, warnings
import numpy as np
import cv2
import torch
import torchvision
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
warnings.filterwarnings('ignore')

from torchvision.models.detection import (
    fasterrcnn_mobilenet_v3_large_320_fpn,
    ssdlite320_mobilenet_v3_large,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.ssd import SSDClassificationHead

# ── Device ────────────────────────────────────────────────────────────────────
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps'); print('🍎 Apple MPS')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda'); print(f'⚡ CUDA: {torch.cuda.get_device_name(0)}')
else:
    DEVICE = torch.device('cpu'); print('🖥️  CPU')

print(f'PyTorch {torch.__version__}  |  Torchvision {torchvision.__version__}')


🍎 Apple MPS
PyTorch 2.10.0  |  Torchvision 0.25.0


## 2 · Paths & Configuration

In [33]:
import os, glob, yaml

# ── Dataset (local) ───────────────────────────────────────────────────────────
DATASET_ROOT = os.path.abspath('./drone-dataset')

# Build a working data.yaml with absolute paths (Ultralytics needs this)
DATA_YAML = os.path.abspath('./drone_local.yaml')
with open(DATA_YAML, 'w') as f:
    yaml.dump({
        'path':  DATASET_ROOT,
        'train': 'train/images',
        'val':   'valid/images',
        'test':  'test/images',
        'nc':    3,
        'names': ['AirPlane', 'Drone', 'Helicopter'],
    }, f)
print(f'✅ data.yaml written → {DATA_YAML}')

# ── Trained weights (project root) ────────────────────────────────────────────
YOLO_WEIGHTS  = os.path.abspath('./drone_yolov8s_final.pt')
DETR_WEIGHTS  = os.path.abspath('./best_detr.pt')
FRCNN_WEIGHTS = os.path.abspath('./fasterrcnn_drone.pth')
SSD_WEIGHTS   = os.path.abspath('./ssd_drone_model_kaggle.pth')

# ── Classes ───────────────────────────────────────────────────────────────────
CLASS_NAMES = ['AirPlane', 'Drone', 'Helicopter']
NUM_CLASSES = len(CLASS_NAMES)   # 3
CONF_THRESH = 0.30
IMG_SIZE    = 640

# Verify everything exists
print('\n📦 Checking weight files:')
for name, path in [('YOLOv8s',      YOLO_WEIGHTS),
                    ('RT-DETR-L',    DETR_WEIGHTS),
                    ('Faster R-CNN', FRCNN_WEIGHTS),
                    ('SSD',          SSD_WEIGHTS)]:
    ok = os.path.exists(path)
    print(f'  {"✅" if ok else "❌ MISSING"}  {name:15s} ({path})')

print(f'\n📂 Dataset root: {DATASET_ROOT}')
test_dir = os.path.join(DATASET_ROOT, 'test/images')
val_dir  = os.path.join(DATASET_ROOT, 'valid/images')
EVAL_DIR = test_dir if os.path.isdir(test_dir) else val_dir
EVAL_LBL = EVAL_DIR.replace('images', 'labels')
test_imgs = sorted(glob.glob(f'{EVAL_DIR}/*.jpg') + glob.glob(f'{EVAL_DIR}/*.png'))
print(f'📷 Eval images: {len(test_imgs)}  ({EVAL_DIR})')


✅ data.yaml written → /Users/hilkin/Development/Drone-Detection/drone_local.yaml

📦 Checking weight files:
  ✅  YOLOv8s         (/Users/hilkin/Development/Drone-Detection/drone_yolov8s_final.pt)
  ✅  RT-DETR-L       (/Users/hilkin/Development/Drone-Detection/best_detr.pt)
  ✅  Faster R-CNN    (/Users/hilkin/Development/Drone-Detection/fasterrcnn_drone.pth)
  ✅  SSD             (/Users/hilkin/Development/Drone-Detection/ssd_drone_model_kaggle.pth)

📂 Dataset root: /Users/hilkin/Development/Drone-Detection/drone-dataset
📷 Eval images: 596  (/Users/hilkin/Development/Drone-Detection/drone-dataset/test/images)


## 3 · Model Architecture Builders

In [34]:
def build_fasterrcnn(num_classes, weights_path=None):
    """MobileNetV3-Large-320-FPN — matches fasterrcnn_drone.pth training arch."""
    model = fasterrcnn_mobilenet_v3_large_320_fpn(weights=None)
    in_f  = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_f, num_classes + 1)
    if weights_path and os.path.exists(weights_path):
        state = torch.load(weights_path, map_location='cpu')
        model.load_state_dict(state)
        print(f'  ✅ Faster R-CNN weights loaded')
    return model.to(DEVICE).eval()


def build_ssd(num_classes, weights_path=None):
    """SSDLite MobileNetV3-Large-320 — matches ssd_drone_model_kaggle.pth arch."""
    model = ssdlite320_mobilenet_v3_large(weights='DEFAULT')
    in_ch = [672, 480, 512, 256, 256, 128]
    n_anc = model.anchor_generator.num_anchors_per_location()
    model.head.classification_head = SSDClassificationHead(in_ch, n_anc, num_classes + 1)
    if weights_path and os.path.exists(weights_path):
        state = torch.load(weights_path, map_location='cpu')
        model.load_state_dict(state)
        print(f'  ✅ SSD weights loaded')
    return model.to(DEVICE).eval()


print('✅ Architecture builders defined')


✅ Architecture builders defined


## 4 · Load All Pre-trained Models

In [35]:
from ultralytics import YOLO, RTDETR

models = {}

# YOLOv8s
if os.path.exists(YOLO_WEIGHTS):
    models['YOLOv8s'] = YOLO(YOLO_WEIGHTS)
    print(f'✅ YOLOv8s loaded      ({YOLO_WEIGHTS})')
else:
    print(f'❌ YOLOv8s not found:  {YOLO_WEIGHTS}')

# RT-DETR-L
if os.path.exists(DETR_WEIGHTS):
    models['RT-DETR-L'] = RTDETR(DETR_WEIGHTS)
    print(f'✅ RT-DETR-L loaded    ({DETR_WEIGHTS})')
else:
    print(f'❌ RT-DETR-L not found: {DETR_WEIGHTS}')

# Faster R-CNN
print('Loading Faster R-CNN...')
models['Faster-RCNN'] = build_fasterrcnn(NUM_CLASSES, FRCNN_WEIGHTS)

# SSD
print('Loading SSD...')
models['SSD'] = build_ssd(NUM_CLASSES, SSD_WEIGHTS)

print(f'\n🎯 {len(models)}/4 models loaded')


✅ YOLOv8s loaded      (/Users/hilkin/Development/Drone-Detection/drone_yolov8s_final.pt)
✅ RT-DETR-L loaded    (/Users/hilkin/Development/Drone-Detection/best_detr.pt)
Loading Faster R-CNN...
  ✅ Faster R-CNN weights loaded
Loading SSD...
  ✅ SSD weights loaded

🎯 4/4 models loaded


## 5 · Validate YOLOv8s & RT-DETR-L
Uses Ultralytics `.val()` on the local dataset — produces mAP50, mAP50-95, Precision, Recall.


In [26]:
val_results = {}

ul_device = 'mps' if DEVICE.type=='mps' else (0 if DEVICE.type=='cuda' else 'cpu')

for mname in ['YOLOv8s', 'RT-DETR-L']:
    if mname not in models:
        print(f'⏭️  {mname} not loaded, skipping')
        continue
    print(f'\n📊 Validating {mname}...')
    metrics = models[mname].val(
        data    = DATA_YAML,
        conf    = CONF_THRESH,
        imgsz   = IMG_SIZE,
        device  = ul_device,
        verbose = False,
    )
    val_results[mname] = {
        'mAP@0.5':    round(metrics.box.map50, 4),
        'mAP@0.5:95': round(metrics.box.map,   4),
        'Precision':  round(metrics.box.mp,    4),
        'Recall':     round(metrics.box.mr,    4),
        'Avg ms':     '—',
        'FPS':        '—',
    }
    print(f'  mAP@0.5={val_results[mname]["mAP@0.5"]}  '
          f'mAP@0.5:95={val_results[mname]["mAP@0.5:95"]}  '
          f'P={val_results[mname]["Precision"]}  R={val_results[mname]["Recall"]}')



📊 Validating YOLOv8s...
Ultralytics 8.4.14 🚀 Python-3.14.2 torch-2.10.0 MPS (Apple M4)
val: Fast image access ✅ (ping: 0.2±0.1 ms, read: 85.9±25.4 MB/s, size: 13.2 KB)
val: Scanning /Users/hilkin/Development/Drone-Detection/drone-dataset/valid/labels... 603 images, 110 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 603/603 7.1Kit/s 0.1s
val: New cache created: /Users/hilkin/Development/Drone-Detection/drone-dataset/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 2.1it/s 18.4s0.3s
                   all        603        497      0.924      0.942      0.946      0.639
Speed: 0.2ms preprocess, 10.6ms inference, 0.0ms loss, 9.7ms postprocess per image
Results saved to /Users/hilkin/Development/Drone-Detection/runs/detect/val11
  mAP@0.5=0.9462  mAP@0.5:95=0.6386  P=0.9236  R=0.942

📊 Validating RT-DETR-L...
Ultralytics 8.4.14 🚀 Python-3.14.2 torch-2.10.0 MPS (Apple M4)
rt-detr-l summary: 313 layers, 31,989,90

## 6 · Evaluate Faster R-CNN & SSD
Computes mAP@0.5 via `torchmetrics` + measures per-image latency.


In [27]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torchmetrics'])
from torchmetrics.detection import MeanAveragePrecision

print(f'Evaluating on {len(test_imgs)} images...')

def parse_gt(lbl_path, W, H):
    boxes, labels = [], []
    if os.path.exists(lbl_path):
        for line in open(lbl_path):
            p = line.strip().split()
            if len(p) < 5: continue
            c, cx, cy, bw, bh = int(p[0]), *map(float, p[1:5])
            boxes.append([(cx-bw/2)*W, (cy-bh/2)*H, (cx+bw/2)*W, (cy+bh/2)*H])
            labels.append(c)
    return boxes, labels


def eval_torchvision(mname, model, img_size=None):
    metric = MeanAveragePrecision(iou_thresholds=[0.5])
    times  = []
    for img_path in test_imgs:
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        if img_size:
            img = cv2.resize(img, (img_size, img_size))
        H, W = img.shape[:2]
        t = torch.from_numpy(img.astype(np.float32)/255.).permute(2,0,1).unsqueeze(0).to(DEVICE)

        t0 = time.perf_counter()
        with torch.no_grad(): pred = model(t)[0]
        times.append((time.perf_counter()-t0)*1000)

        keep = pred['scores'].cpu() >= CONF_THRESH
        lbl_path = os.path.join(EVAL_LBL,
                   os.path.splitext(os.path.basename(img_path))[0] + '.txt')
        gt_b, gt_l = parse_gt(lbl_path, W, H)

        metric.update(
            [{'boxes':  pred['boxes'].cpu()[keep],
              'scores': pred['scores'].cpu()[keep],
              'labels': (pred['labels'].cpu()[keep]-1).clamp(min=0)}],
            [{'boxes':  torch.tensor(gt_b, dtype=torch.float32) if gt_b else torch.zeros((0,4)),
              'labels': torch.tensor(gt_l, dtype=torch.long)}],
        )

    r = metric.compute()
    avg_ms = float(np.mean(times))
    val_results[mname] = {
        'mAP@0.5':    round(r['map_50'].item(), 4),
        'mAP@0.5:95': round(r['map'].item(),    4),
        'Precision':  '—',
        'Recall':     '—',
        'Avg ms':     round(avg_ms, 1),
        'FPS':        round(1000/avg_ms, 1),
    }
    print(f'  {mname}: mAP@0.5={val_results[mname]["mAP@0.5"]}  '
          f'avg={avg_ms:.1f}ms  FPS={val_results[mname]["FPS"]}')


for mname, key, sz in [('Faster-RCNN','Faster-RCNN',None), ('SSD','SSD',320)]:
    if key in models:
        print(f'\n📊 Evaluating {mname}...')
        eval_torchvision(mname, models[key], img_size=sz)
    else:
        print(f'⏭️  {mname} not loaded')


Evaluating on 596 images...

📊 Evaluating Faster-RCNN...
  Faster-RCNN: mAP@0.5=0.8893  avg=193.8ms  FPS=5.2

📊 Evaluating SSD...
  SSD: mAP@0.5=0.7396  avg=29.3ms  FPS=34.1


## 7 · Results Summary Table

In [28]:
if val_results:
    df = pd.DataFrame(val_results).T
    df.index.name = 'Model'

    def highlight_best(s):
        vals = pd.to_numeric(s, errors='coerce')
        if vals.isna().all(): return [''] * len(s)
        best = vals.idxmax()
        return ['background-color:#166534;color:white;font-weight:bold'
                if i == best else '' for i in s.index]

    display(
        df.style
        .apply(highlight_best)
        .format(na_rep='—')
        .set_caption('🚁 Drone Detection — Pre-trained Model Evaluation (Green = best)')
        .set_table_styles([
            {'selector':'caption','props':[('font-size','1.15em'),('font-weight','bold'),('margin-bottom','8px')]},
            {'selector':'th','props':[('background','#1e293b'),('color','white'),('padding','7px 12px')]},
            {'selector':'td','props':[('padding','5px 12px'),('text-align','center')]},
        ])
    )
    df.reset_index().to_csv('./eval_results_local.csv', index=False)
    print('💾 eval_results_local.csv saved')
else:
    print('No results yet — run cells 5 & 6 first.')


,mAP@0.5,mAP@0.5:95,Precision,Recall,Avg ms,FPS
Model,,,,,,
YOLOv8s,0.946200,0.638600,0.923600,0.942000,—,—
RT-DETR-L,0.655800,0.350800,0.773300,0.574600,—,—
Faster-RCNN,0.889300,0.889300,—,—,193.800000,5.200000
SSD,0.739600,0.739600,—,—,29.300000,34.100000


💾 eval_results_local.csv saved


## 8 · Side-by-Side Inference Demo
Runs all 4 models on the same random test images and draws bounding boxes.


In [29]:
COLORS = {
    'YOLOv8s':    (52,  211, 153),   # green
    'RT-DETR-L':  (251, 146,  60),   # orange
    'Faster-RCNN':(99,  102, 241),   # indigo
    'SSD':        (236,  72, 153),   # pink
}

def draw_tv(img, model, color, sz=None):
    """Run a Torchvision model and draw boxes on img."""
    h, w = img.shape[:2]
    inp  = img if sz is None else cv2.resize(img, (sz, sz))
    ih, iw = inp.shape[:2]
    t = torch.from_numpy(inp.astype(np.float32)/255.).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): pred = model(t)[0]
    vis = img.copy()
    for box, score, lbl in zip(pred['boxes'].cpu(), pred['scores'].cpu(), pred['labels'].cpu()):
        if score < CONF_THRESH: continue
        x1, y1, x2, y2 = box.tolist()
        if sz: x1,x2=x1*w/iw,x2*w/iw; y1,y2=y1*h/ih,y2*h/ih
        cls = lbl.item() - 1
        nm  = CLASS_NAMES[cls] if 0 <= cls < NUM_CLASSES else '?'
        cv2.rectangle(vis, (int(x1),int(y1)), (int(x2),int(y2)), color, 2)
        cv2.putText(vis, f'{nm} {score:.2f}', (int(x1), max(int(y1)-6, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    return vis


sample = random.sample(test_imgs, min(4, len(test_imgs)))
n_m    = len(models)
n_img  = len(sample)

fig, axes = plt.subplots(n_img, n_m, figsize=(5*n_m, 4*n_img))
if n_img == 1: axes = [axes]
if n_m   == 1: axes = [[ax] for ax in axes]

ul_dev = 'mps' if DEVICE.type=='mps' else (0 if DEVICE.type=='cuda' else 'cpu')

for row, img_path in enumerate(sample):
    base = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    for col, (mname, model) in enumerate(models.items()):
        ax    = axes[row][col]
        color = COLORS.get(mname, (255,255,0))
        if mname in ('YOLOv8s', 'RT-DETR-L'):
            r   = model.predict(img_path, conf=CONF_THRESH, imgsz=IMG_SIZE,
                                device=ul_dev, verbose=False)[0]
            vis = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
        else:
            sz  = 320 if mname == 'SSD' else None
            vis = draw_tv(base, model, color, sz=sz)
        ax.imshow(vis); ax.axis('off')
        if row == 0:
            ax.set_title(mname, fontsize=13, fontweight='bold')

plt.suptitle('Side-by-Side Inference — All 4 Models', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('./inference_comparison.png', bbox_inches='tight', dpi=120)
plt.show()
print('💾 inference_comparison.png saved')


<Figure size 2000x1600 with 16 Axes>

💾 inference_comparison.png saved


## 9 · Latency & FPS Comparison

## 🧬 Multi-Model Ensemble — The Real Purpose

The goal of this notebook is **not** just to evaluate models separately — it's to **combine** them.

### What is an Ensemble?
Run multiple models on the same image, then **merge** their bounding box predictions.
This exploits the fact that each model has different strengths:

| Model | Strength | Role in Ensemble |
|-------|----------|------------------|
| **YOLOv8s** | Fast, high confidence aerial objects | Primary detector |
| **RT-DETR-L** | Transformer attention — catches occluded/distant objects | Accuracy booster |
| **Faster R-CNN** | Strong 2-stage proposals | False-positive reducer |
| **SSD** | Catches small objects at multi-scale feature maps | Small object specialist |

### Strategy: Weighted Box Fusion (WBF)
WBF is better than simple NMS for ensembles — instead of discarding overlapping boxes,
it **averages** them weighted by their confidence scores. Result: tighter, more confident boxes.

```
Image → [YOLOv8s predictions]   ─┐
Image → [RT-DETR predictions]   ─┤→ WBF merge → Final ensemble detections
Image → [Faster R-CNN preds]   ─┤
Image → [SSD predictions]      ─┘
```
> You can also run a **speed-accuracy cascade**: use SSD first (28ms), escalate to YOLO,
> then RT-DETR only for low-confidence images. Both modes are implemented below.


In [36]:
# ── Weighted Box Fusion ensemble ─────────────────────────────────────────────
# WBF merges overlapping boxes from all models by averaging their coordinates
# weighted by confidence — much better than NMS for multi-model ensembles.
import time as _time
import numpy as np

def iou(b1, b2):
    """Compute IoU between two boxes [x1,y1,x2,y2]."""
    xi1,yi1 = max(b1[0],b2[0]), max(b1[1],b2[1])
    xi2,yi2 = min(b1[2],b2[2]), min(b1[3],b2[3])
    inter = max(0,xi2-xi1)*max(0,yi2-yi1)
    a1 = (b1[2]-b1[0])*(b1[3]-b1[1])
    a2 = (b2[2]-b2[0])*(b2[3]-b2[1])
    return inter/(a1+a2-inter+1e-6)


def wbf(all_boxes, all_scores, all_labels, iou_thr=0.50, skip_box_thr=0.01):
    """
    Weighted Box Fusion.
    all_boxes  : list of np.ndarray (N_i, 4)  one per model
    all_scores : list of np.ndarray (N_i,)
    all_labels : list of np.ndarray (N_i,)  int
    Returns fused (boxes, scores, labels).
    """
    # Flatten all predictions
    boxes  = np.concatenate(all_boxes,  axis=0) if any(len(b)>0 for b in all_boxes) else np.zeros((0,4))
    scores = np.concatenate(all_scores, axis=0) if any(len(s)>0 for s in all_scores) else np.zeros(0)
    labels = np.concatenate(all_labels, axis=0) if any(len(l)>0 for l in all_labels) else np.zeros(0,int)

    if len(boxes) == 0:
        return np.zeros((0,4)), np.zeros(0), np.zeros(0,int)

    keep = scores >= skip_box_thr
    boxes, scores, labels = boxes[keep], scores[keep], labels[keep]

    order = np.argsort(-scores)
    boxes, scores, labels = boxes[order], scores[order], labels[order]

    clusters_b, clusters_s, clusters_l = [], [], []

    for i in range(len(boxes)):
        matched = -1
        for j,(cb,_,cl) in enumerate(zip(clusters_b, clusters_s, clusters_l)):
            rep_box = np.average(cb, axis=0, weights=_)
            if cl[0] == labels[i] and iou(rep_box, boxes[i]) >= iou_thr:
                matched = j; break
        if matched >= 0:
            clusters_b[matched].append(boxes[i])
            clusters_s[matched].append(scores[i])
            clusters_l[matched].append(labels[i])
        else:
            clusters_b.append([boxes[i]])
            clusters_s.append([scores[i]])
            clusters_l.append([labels[i]])

    out_boxes, out_scores, out_labels = [], [], []
    for cb, cs, cl in zip(clusters_b, clusters_s, clusters_l):
        w = np.array(cs)
        out_boxes.append(np.average(cb, axis=0, weights=w))
        out_scores.append(float(np.mean(w)))   # mean score across models that saw it
        out_labels.append(int(np.bincount(cl).argmax()))

    return (np.array(out_boxes), np.array(out_scores), np.array(out_labels,int))


print('✅ WBF ensemble function defined')


✅ WBF ensemble function defined


### Ensemble Inference on a Single Image
Runs all 4 models, merges predictions with WBF, shows the result.


In [37]:
def predict_ultralytics(model, img_path, conf, device):
    r = model.predict(img_path, conf=conf, imgsz=640, device=device, verbose=False)[0]
    if r.boxes and len(r.boxes):
        return (r.boxes.xyxy.cpu().numpy(),
                r.boxes.conf.cpu().numpy(),
                r.boxes.cls.cpu().numpy().astype(int))
    return np.zeros((0,4)), np.zeros(0), np.zeros(0,int)


def predict_torchvision(model, img, device, conf, sz=None):
    h,w = img.shape[:2]
    inp = img if sz is None else cv2.resize(img,(sz,sz))
    ih,iw = inp.shape[:2]
    t = torch.from_numpy(inp.astype(np.float32)/255.).permute(2,0,1).unsqueeze(0).to(device)
    with torch.no_grad(): pred = model(t)[0]
    mask = pred['scores'].cpu().numpy() >= conf
    boxes  = pred['boxes'].cpu().numpy()[mask]
    scores = pred['scores'].cpu().numpy()[mask]
    labels = (pred['labels'].cpu().numpy()[mask]-1).clip(min=0)  # remove BG
    # scale back to original size
    if sz and len(boxes):
        boxes[:,0] *= w/iw; boxes[:,2] *= w/iw
        boxes[:,1] *= h/ih; boxes[:,3] *= h/ih
    return boxes, scores, labels


def ensemble_predict(img_path, conf=CONF_THRESH, iou_thr=0.45):
    """
    Run all loaded models on img_path and fuse with WBF.
    Returns (boxes, scores, labels, total_ms).
    """
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    ul_dev = 'mps' if DEVICE.type=='mps' else (0 if DEVICE.type=='cuda' else 'cpu')

    all_b, all_s, all_l = [], [], []
    t0 = _time.perf_counter()

    for mname, model in models.items():
        if mname in ('YOLOv8s','RT-DETR-L'):
            b,s,l = predict_ultralytics(model, img_path, conf, ul_dev)
        else:
            sz = 320 if mname=='SSD' else None
            b,s,l = predict_torchvision(model, img, DEVICE, conf, sz)
        all_b.append(b); all_s.append(s); all_l.append(l)

    total_ms = (_time.perf_counter()-t0)*1000
    fused_b, fused_s, fused_l = wbf(all_b, all_s, all_l, iou_thr=iou_thr)
    return fused_b, fused_s, fused_l, total_ms, img


print('✅ Ensemble predict functions defined')


✅ Ensemble predict functions defined


### Visual Demo — Single image, all models vs. ensemble


In [38]:
sample = random.choice(test_imgs)
img_base = cv2.cvtColor(cv2.imread(sample), cv2.COLOR_BGR2RGB)
H, W = img_base.shape[:2]
ul_dev = 'mps' if DEVICE.type=='mps' else (0 if DEVICE.type=='cuda' else 'cpu')

COLORS = {'YOLOv8s':(52,211,153),'RT-DETR-L':(251,146,60),
          'Faster-RCNN':(99,102,241),'SSD':(236,72,153)}

def draw(img, boxes, scores, labels, color, title):
    vis = img.copy()
    for box,score,lbl in zip(boxes,scores,labels):
        nm  = CLASS_NAMES[int(lbl)] if 0<=int(lbl)<NUM_CLASSES else '?'
        cv2.rectangle(vis,(int(box[0]),int(box[1])),(int(box[2]),int(box[3])),color,2)
        cv2.putText(vis,f'{nm} {score:.2f}',(int(box[0]),max(int(box[1])-6,10)),
                    cv2.FONT_HERSHEY_SIMPLEX,0.55,color,2)
    return vis

# Collect individual model predictions
individual = {}
for mname, model in models.items():
    color = COLORS.get(mname,(255,255,0))
    if mname in ('YOLOv8s','RT-DETR-L'):
        b,s,l = predict_ultralytics(model, sample, CONF_THRESH, ul_dev)
    else:
        sz = 320 if mname=='SSD' else None
        b,s,l = predict_torchvision(model, img_base, DEVICE, CONF_THRESH, sz)
    individual[mname] = (b,s,l,color)

# Ensemble prediction
ens_b, ens_s, ens_l, ens_ms, _ = ensemble_predict(sample)

# Plot: top row = individual models, bottom = ensemble
n = len(individual)
fig, axes = plt.subplots(2, max(n,1), figsize=(5*max(n,1), 9))
if n == 1: axes = [[axes[0]], [axes[1]]]

for col,(mname,(b,s,l,color)) in enumerate(individual.items()):
    vis = draw(img_base, b, s, l, color, mname)
    axes[0][col].imshow(vis); axes[0][col].axis('off')
    axes[0][col].set_title(f'{mname}\n({len(b)} det)', fontsize=11, fontweight='bold')

# Ensemble in bottom-centre
ens_vis = draw(img_base, ens_b, ens_s, ens_l, (255,215,0), 'Ensemble')
for col in range(n):
    axes[1][col].axis('off')
mid = n//2
axes[1][mid].imshow(ens_vis); axes[1][mid].axis('on')
axes[1][mid].set_title(
    f'🧬 WBF Ensemble Result\n({len(ens_b)} fused detections  •  {ens_ms:.0f}ms total)',
    fontsize=13, fontweight='bold', color='#92400e')
axes[1][mid].axis('off')

plt.suptitle('Individual Models  vs.  WBF Ensemble', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('./ensemble_demo.png', bbox_inches='tight', dpi=120)
plt.show()
print(f'💾 ensemble_demo.png saved  |  Ensemble: {len(ens_b)} detections in {ens_ms:.0f}ms')


<Figure size 2000x900 with 8 Axes>

💾 ensemble_demo.png saved  |  Ensemble: 1 detections in 132ms


### Speed-Accuracy Cascade (Alternative Ensemble Mode)

Instead of running all 4 models every time, use a **cascade**:
- **Step 1:** Run SSD (fastest, ~28ms). If max confidence ≥ 0.7 → return immediately.
- **Step 2:** If uncertain, also run YOLOv8s and fuse with WBF.
- **Step 3:** If still below threshold, add RT-DETR for the hardest cases.

This gives near-RT-DETR accuracy for easy images but at SSD-level speed.


In [39]:
def cascade_predict(img_path, fast_conf=0.70, slow_conf=0.30):
    """
    Speed-accuracy cascade:
      Stage 1 (SSD  ~28ms): if confident enough → done
      Stage 2 (YOLO ~64ms): fuse with stage-1, if confident enough → done
      Stage 3 (RT-DETR)   : fuse all three for the hardest cases
    Returns (boxes, scores, labels, ms, stages_used).
    """
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    ul_dev = 'mps' if DEVICE.type=='mps' else (0 if DEVICE.type=='cuda' else 'cpu')
    t0 = _time.perf_counter()

    preds = {}  # mname → (boxes, scores, labels)

    # Stage 1 — SSD
    if 'SSD' in models:
        b,s,l = predict_torchvision(models['SSD'], img, DEVICE, slow_conf, sz=320)
        preds['SSD'] = (b,s,l)
        if len(s) and s.max() >= fast_conf:
            ms = (_time.perf_counter()-t0)*1000
            return b, s, l, ms, ['SSD']

    # Stage 2 — YOLO
    if 'YOLOv8s' in models:
        b,s,l = predict_ultralytics(models['YOLOv8s'], img_path, slow_conf, ul_dev)
        preds['YOLOv8s'] = (b,s,l)
        # Fuse stage 1+2
        all_b = [v[0] for v in preds.values()]
        all_s = [v[1] for v in preds.values()]
        all_l = [v[2] for v in preds.values()]
        fb,fs,fl = wbf(all_b, all_s, all_l)
        if len(fs) and fs.max() >= fast_conf:
            ms = (_time.perf_counter()-t0)*1000
            return fb, fs, fl, ms, list(preds.keys())

    # Stage 3 — RT-DETR (heaviest, only for hard cases)
    if 'RT-DETR-L' in models:
        b,s,l = predict_ultralytics(models['RT-DETR-L'], img_path, slow_conf, ul_dev)
        preds['RT-DETR-L'] = (b,s,l)
        all_b = [v[0] for v in preds.values()]
        all_s = [v[1] for v in preds.values()]
        all_l = [v[2] for v in preds.values()]
        fb,fs,fl = wbf(all_b, all_s, all_l)
        ms = (_time.perf_counter()-t0)*1000
        return fb, fs, fl, ms, list(preds.keys())

    ms = (_time.perf_counter()-t0)*1000
    return np.zeros((0,4)), np.zeros(0), np.zeros(0,int), ms, []


# --- Run cascade on 20 random images and report average stages used ---
if len(test_imgs) >= 5:
    bench = random.sample(test_imgs, min(20, len(test_imgs)))
    stage_counts = {1:0, 2:0, 3:0}
    times = []
    for p in bench:
        _,_,_,ms,stages = cascade_predict(p)
        times.append(ms)
        stage_counts[len(stages)] = stage_counts.get(len(stages),0)+1

    print(f'\n⚡ Cascade Benchmark ({len(bench)} images):')
    print(f'  Avg latency : {np.mean(times):.1f}ms')
    print(f'  Stopped at SSD only (stage 1)  : {stage_counts.get(1,0)} images')
    print(f'  Needed YOLO  (stage 2)         : {stage_counts.get(2,0)} images')
    print(f'  Needed RT-DETR (stage 3)       : {stage_counts.get(3,0)} images')
    print(f'\n  → Most images resolved at stage 1-2 (fast path)')
    print(f'    vs full ensemble avg: ~{sum(times)/len(times):.0f}ms')



⚡ Cascade Benchmark (20 images):
  Avg latency : 67.1ms
  Stopped at SSD only (stage 1)  : 10 images
  Needed YOLO  (stage 2)         : 2 images
  Needed RT-DETR (stage 3)       : 8 images

  → Most images resolved at stage 1-2 (fast path)
    vs full ensemble avg: ~67ms


In [40]:
import time as _time

N_BENCH = 30   # images to time per model
bench_imgs = random.sample(test_imgs, min(N_BENCH, len(test_imgs)))
ul_dev = 'mps' if DEVICE.type=='mps' else (0 if DEVICE.type=='cuda' else 'cpu')
latency = {}

for mname, model in models.items():
    times = []
    for p in bench_imgs:
        if mname in ('YOLOv8s','RT-DETR-L'):
            t0 = _time.perf_counter()
            model.predict(p, conf=CONF_THRESH, imgsz=IMG_SIZE, device=ul_dev, verbose=False)
            times.append((_time.perf_counter()-t0)*1000)
        else:
            img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
            sz  = 320 if mname=='SSD' else None
            if sz: img = cv2.resize(img,(sz,sz))
            t = torch.from_numpy(img.astype(np.float32)/255.).permute(2,0,1).unsqueeze(0).to(DEVICE)
            t0 = _time.perf_counter()
            with torch.no_grad(): model(t)
            times.append((_time.perf_counter()-t0)*1000)
    latency[mname] = np.mean(times)
    print(f'  {mname:15s}  avg={latency[mname]:.1f}ms  FPS={1000/latency[mname]:.1f}')

fig, (ax1,ax2) = plt.subplots(1, 2, figsize=(12,4))
names = list(latency.keys())
lats  = [latency[n] for n in names]
fps   = [1000/l for l in lats]
colors= ['#4ade80','#fb923c','#818cf8','#f472b6']

ax1.bar(names, lats, color=colors, edgecolor='white', linewidth=0.8)
ax1.set(title='Avg Latency (ms) ↓ lower is better', ylabel='ms')
ax1.spines[['top','right']].set_visible(False)
for i,(n,v) in enumerate(zip(names,lats)): ax1.text(i, v+0.5, f'{v:.1f}', ha='center', fontsize=10)

ax2.bar(names, fps, color=colors, edgecolor='white', linewidth=0.8)
ax2.set(title='FPS ↑ higher is better', ylabel='FPS')
ax2.spines[['top','right']].set_visible(False)
for i,(n,v) in enumerate(zip(names,fps)): ax2.text(i, v+0.3, f'{v:.1f}', ha='center', fontsize=10)

plt.suptitle(f'Inference Speed  (avg over {N_BENCH} images)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('./latency_comparison.png', bbox_inches='tight', dpi=120)
plt.show()


  YOLOv8s          avg=21.9ms  FPS=45.6
  RT-DETR-L        avg=70.3ms  FPS=14.2
  Faster-RCNN      avg=123.1ms  FPS=8.1
  SSD              avg=16.8ms  FPS=59.5


<Figure size 1200x400 with 2 Axes>

## 10 · Architecture Comparison Table

In [41]:
rows = [
    {'Model':'YOLOv8s','Backbone':'CSP-Darknet C2f','Neck':'FPN+PAN',
     'Head':'Anchor-free Decoupled','Params':'~11.1M','Input':'640×640','Notes':'Fastest; real-time'},
    {'Model':'RT-DETR-L','Backbone':'ResNet-50+HybridEnc','Neck':'Transformer',
     'Head':'Bipartite Matching (no NMS)','Params':'~32M','Input':'640×640','Notes':'Highest accuracy'},
    {'Model':'Faster R-CNN','Backbone':'MobileNetV3-Large','Neck':'FPN-320',
     'Head':'RPN + RoI-Align + FastRCNN','Params':'~19.4M','Input':'Variable','Notes':'Strong 2-stage'},
    {'Model':'SSD','Backbone':'MobileNetV3-Large','Neck':'SSDLite-320',
     'Head':'SSD Classification','Params':'~3.4M','Input':'320×320','Notes':'Lightest; edge-ready'},
]
df_arch = pd.DataFrame(rows).set_index('Model')
display(df_arch.style
    .set_caption('🏗️ Architecture Comparison — All 4 Drone Detection Models')
    .set_table_styles([
        {'selector':'caption','props':[('font-size','1.15em'),('font-weight','bold'),('margin-bottom','8px')]},
        {'selector':'th','props':[('background','#1e293b'),('color','white'),('padding','6px 12px')]},
        {'selector':'td','props':[('padding','5px 12px')]},
    ])
)


,Backbone,Neck,Head,Params,Input,Notes
Model,,,,,,
YOLOv8s,CSP-Darknet C2f,FPN+PAN,Anchor-free Decoupled,~11.1M,640×640,Fastest; real-time
RT-DETR-L,ResNet-50+HybridEnc,Transformer,Bipartite Matching (no NMS),~32M,640×640,Highest accuracy
Faster R-CNN,MobileNetV3-Large,FPN-320,RPN + RoI-Align + FastRCNN,~19.4M,Variable,Strong 2-stage
SSD,MobileNetV3-Large,SSDLite-320,SSD Classification,~3.4M,320×320,Lightest; edge-ready
